# 05 — Limpieza y preparación ENOE → Parquet (Plata)

**Propósito:** Leer el Parquet bronce de ENOE, aplicar limpieza,
castings, tratamiento de nulos y duplicados, y generar la capa Plata
lista para consumo analítico.

**Pipeline:** `ingesta_general.ipynb` → **`05_limpieza_enoe.ipynb`** → `06_integracion.ipynb`

**Prerequisito:** haber ejecutado `ingesta_general.ipynb` completo.

In [ ]:
# ── Importaciones ───────────────────────────────────────────────────────────
import os
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, DoubleType, StringType, ShortType
)

spark = (
    SparkSession.builder
    .appName('05_limpieza_enoe')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.parquet.compression.codec', 'snappy')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark', spark.version)

In [ ]:
# ── Rutas ───────────────────────────────────────────────────────────────────
BASE        = Path(os.getcwd()).parent          # raíz del repositorio
RAW         = BASE / 'data' / 'raw'
SILVER      = BASE / 'data' / 'silver'         # capa Plata

ENOE_PARQUET       = str(RAW    / 'enoe' / 'parquet')   # bronce (entrada)
ENOE_SILVER        = str(SILVER / 'enoe')               # plata  (salida)

Path(ENOE_SILVER).mkdir(parents=True, exist_ok=True)

print('Bronce :', ENOE_PARQUET)
print('Plata  :', ENOE_SILVER)

---
## 1 · Extracción — leer Parquet bronce

Se lee el Parquet generado por `ingesta_general.ipynb` (capa Bronce).
Se registran conteos y esquema de entrada para evidencia.

In [ ]:
# ── 1.1 Leer Parquet bronce ─────────────────────────────────────────────────
df_bronce = spark.read.parquet(ENOE_PARQUET)

filas_entrada = df_bronce.count()
print(f'Filas entrada  : {filas_entrada:,}')
print(f'Columnas       : {len(df_bronce.columns)}')
print(f'Particiones    : {df_bronce.rdd.getNumPartitions()}')
df_bronce.printSchema()

In [ ]:
# ── 1.2 Cobertura por año y trimestre (entrada) ──────────────────────────────
(
    df_bronce
    .groupBy('anio', 'trimestre')
    .agg(F.count('*').alias('filas'))
    .orderBy('anio', 'trimestre')
    .show(100, truncate=False)
)

---
## 2 · Validación de entrada

Antes de limpiar, se documentan los problemas encontrados:
nulos por columna, duplicados en llaves de join y valores fuera de rango.
Esto cumple con el criterio de evidencia de la Actividad 11.

In [ ]:
# ── 2.1 Nulos por columna ───────────────────────────────────────────────────
COLS_CLAVE = [
    'sex', 'eda', 'clase1', 'clase2', 'pos_ocu',
    'ing7c', 'ingocup', 'hrsocup', 'fac', 'anio', 'trimestre'
]

null_counts = df_bronce.select([
    F.count(F.when(F.col(c).isNull() | (F.col(c).cast('string') == ''), c))
     .alias(c)
    for c in COLS_CLAVE
])

print('Nulos / vacíos por columna clave:')
null_counts.show(truncate=False)

In [ ]:
# ── 2.2 Duplicados en llaves de join ────────────────────────────────────────
JOIN_KEYS = [
    'r_def', 'cd_a', 'ent', 'con', 'upm', 'd_sem',
    'n_pro_viv', 'v_sel', 'n_hog', 'h_mud', 'n_ent', 'per', 'n_ren'
]

total      = df_bronce.count()
distintos  = df_bronce.dropDuplicates(JOIN_KEYS + ['anio', 'trimestre']).count()
duplicados = total - distintos

print(f'Total filas          : {total:,}')
print(f'Filas únicas (llaves): {distintos:,}')
print(f'Duplicados detectados: {duplicados:,}')

In [ ]:
# ── 2.3 Rangos de variables clave ───────────────────────────────────────────
(
    df_bronce.select(
        F.min('eda').alias('eda_min'),   F.max('eda').alias('eda_max'),
        F.min('ing7c').alias('ing_min'), F.max('ing7c').alias('ing_max'),
        F.min('hrsocup').alias('hrs_min'), F.max('hrsocup').alias('hrs_max'),
        F.min('anio').alias('anio_min'), F.max('anio').alias('anio_max'),
    ).show(truncate=False)
)

---
## 3 · Limpieza y preparación

Se aplican las transformaciones en este orden:
1. Casting a tipos correctos
2. Eliminar duplicados por llaves compuestas
3. Filtrar registros con `eda < 15` (menores fuera de población objetivo ENOE 15+)
4. Tratar nulos en variables de ingreso y horas (`-1` = no aplica en INEGI)
5. Crear columna `periodo` para facilitar joins temporales

> **Decisión justificada:** los registros con `eda < 15` se eliminan porque
> la ENOE 15 y más delimita su universo a personas de 15 años en adelante;
> su presencia en el bronce indica filas de hogares, no de personas.

In [ ]:
# ── 3.1 Casting de tipos ────────────────────────────────────────────────────
# Variables enteras
COLS_INT = [
    'eda', 'sex', 'niv_ins', 'clase1', 'clase2', 'pos_ocu',
    'seg_soc', 'ur', 'zona', 'sub_o', 'tip_con',
    'p1', 'p3', 'p4', 'p4b', 'p4c', 'p5h'
]
# Variables numéricas continuas
COLS_DBL = ['ing7c', 'ingocup', 'hrsocup', 'salario', 'fac']
# Variables string que llegan como número
COLS_STR = ['ent', 'con', 'upm', 'rama', 'scian', 'cs_p13_1', 'anios_esc']

df = df_bronce
for c in COLS_INT:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast(IntegerType()))
for c in COLS_DBL:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast(DoubleType()))

print('Castings aplicados OK')

In [ ]:
# ── 3.2 Eliminar duplicados por llaves compuestas ───────────────────────────
# Se conserva el primer registro para cada combinación de llaves + periodo
df = df.dropDuplicates(JOIN_KEYS + ['anio', 'trimestre'])
print(f'Filas tras dedup: {df.count():,}')

In [ ]:
# ── 3.3 Filtrar menores de 15 años (fuera del universo ENOE 15+) ────────────
antes = df.count()
df = df.filter(F.col('eda') >= 15)
despues = df.count()
print(f'Eliminados eda < 15 : {antes - despues:,} registros')
print(f'Filas restantes     : {despues:,}')

In [ ]:
# ── 3.4 Tratar nulos en ingresos y horas ────────────────────────────────────
# En ENOE, el código -1 indica 'no aplica' (ej. desocupados no tienen ingreso).
# Se convierte -1 a NULL para que las agregaciones no distorsionen promedios.
for c in ['ing7c', 'ingocup', 'hrsocup', 'salario']:
    if c in df.columns:
        df = df.withColumn(
            c,
            F.when(F.col(c) < 0, F.lit(None)).otherwise(F.col(c))
        )

print('Nulos por código -1 reemplazados OK')

In [ ]:
# ── 3.5 Crear columna 'periodo' (YYYYQN) para joins ────────────────────────
# Ejemplo: 2023 T2 → '2023Q2'
df = df.withColumn(
    'periodo',
    F.concat(F.col('anio').cast(StringType()), F.lit('Q'), F.col('trimestre').cast(StringType()))
)

print('Columna periodo creada. Muestra:')
df.select('anio', 'trimestre', 'periodo').distinct().orderBy('anio', 'trimestre').show(10)

---
## 4 · Validación post-limpieza

Se verifica que el dataset limpio cumple las reglas de calidad
antes de escribirlo como Parquet Plata.

In [ ]:
# ── 4.1 Revalidar nulos en columnas clave ───────────────────────────────────
COLS_NO_NULL = ['eda', 'sex', 'clase1', 'anio', 'trimestre']

null_post = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in COLS_NO_NULL
])

print('Nulos post-limpieza en columnas obligatorias (esperado: 0 en todas):')
null_post.show(truncate=False)

In [ ]:
# ── 4.2 Verificar rango de edad ─────────────────────────────────────────────
(
    df.select(
        F.min('eda').alias('eda_min'),
        F.max('eda').alias('eda_max'),
        F.count(F.when(F.col('eda') < 15, True)).alias('menores_15')
    ).show()
)
# Esperado: eda_min >= 15, menores_15 = 0

In [ ]:
# ── 4.3 Distribución de clase de actividad ──────────────────────────────────
# clase1: 1=ocupados, 2=desocupados, 3=no económicamente activos
print('Distribución clase1 (actividad económica):')
(
    df.groupBy('clase1')
    .agg(
        F.count('*').alias('filas'),
        F.round(F.count('*') * 100 / df.count(), 2).alias('pct')
    )
    .orderBy('clase1')
    .show()
)

In [ ]:
# ── 4.4 Resumen de transformaciones ─────────────────────────────────────────
filas_salida = df.count()
print('=' * 55)
print('RESUMEN DE LIMPIEZA — ENOE')
print('=' * 55)
print(f'Filas entrada (bronce) : {filas_entrada:,}')
print(f'Filas salida  (plata)  : {filas_salida:,}')
print(f'Registros eliminados   : {filas_entrada - filas_salida:,}')
print(f'Columnas               : {len(df.columns)}')
print(f'Periodo cubierto       : {df.agg(F.min("anio")).collect()[0][0]}'
      f' – {df.agg(F.max("anio")).collect()[0][0]}')
print('=' * 55)
print('Decisiones de limpieza:')
print('  1. Duplicados por llaves compuestas + periodo eliminados')
print('  2. eda < 15 eliminados (fuera de universo ENOE 15+)')
print('  3. Ingresos/horas con valor -1 → NULL (código INEGI no-aplica)')
print('  4. Tipos normalizados: enteros y doubles donde corresponde')
print('  5. Columna periodo YYYYQN creada para joins')

---
## 5 · Almacenamiento — Parquet Plata

Se escribe el dataset limpio como Parquet en la capa Plata,
particionado por `anio` para mantener consistencia con la capa Bronce.

In [ ]:
# ── 5.1 Escribir Parquet Plata ──────────────────────────────────────────────
(
    df
    .write
    .mode('overwrite')
    .partitionBy('anio')
    .parquet(ENOE_SILVER)
)

size_mb = sum(
    f.stat().st_size for f in Path(ENOE_SILVER).rglob('*.parquet')
) / 1_048_576
print(f'Parquet Plata escrito en : {ENOE_SILVER}')
print(f'Tamaño estimado          : {size_mb:.1f} MB (snappy)')

In [ ]:
# ── 5.2 Validación de lectura (round-trip) ──────────────────────────────────
# Verificar que el Parquet puede volver a leerse correctamente
df_check = spark.read.parquet(ENOE_SILVER)

print('Validación round-trip:')
print(f'  Filas releídas : {df_check.count():,}')
print(f'  Columnas       : {len(df_check.columns)}')
df_check.printSchema()

print('\nCobertura por año (Plata):')
(
    df_check
    .groupBy('anio', 'trimestre')
    .agg(F.count('*').alias('filas'))
    .orderBy('anio', 'trimestre')
    .show(100, truncate=False)
)

In [ ]:
# ── Fin ─────────────────────────────────────────────────────────────────────
print('05_limpieza_enoe.ipynb completado.')
print('Siguiente paso: 06_integracion.ipynb')
spark.stop()